# sre-gym — GRPO on-policy training against the live env

Open in Colab:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dakshdoesdev/sre-enginnerllm/blob/main/train/grpo_run.ipynb)

**Run AFTER `train/sanity_run.ipynb`.** Loads the SFT checkpoint from `dakshdoesdev/sre-gym-qwen35-4b-sft`, runs ~300 GRPO steps where each step samples a scenario, rolls out K=4 trajectories with the current policy, and updates weights on group-relative advantages.

**Reward = env.evaluate()['score']** — deterministic scalar from the grader. No PRM, no LLM-as-judge.

**Before running**:
- Same secrets as `sanity_run.ipynb` (`HFTOKEN`, optional `WANDB_API_KEY`)
- Runtime → A100 GPU
- The notebook auto-clones the repo and boots the env's pool server in-Colab

**Architecture**

```
Colab A100 40GB
├── uvicorn: pool_server on :8100  (env instances + lease TTL)
└── this notebook:
    ├── FastLanguageModel.from_pretrained(SFT_ADAPTER_REPO, load_in_4bit=True)
    ├── GRPO rollout loop:
    │     for step in range(NUM_STEPS):
    │         scenario = sample_scenario()
    │         for k in range(K):
    │             rollout = play_episode(policy, pool_server, scenario)
    │             advantage[k] = rollout.final_score - group_mean
    │         policy_gradient_update(model, rollouts, advantages)
    └── push_to_hub('dakshdoesdev/sre-gym-qwen35-4b-grpo') every 25 steps
```

Expected wall-clock: ~4–6h. Expected GPU memory: ~24 GB.

## 0. Runtime + deps

In [ ]:
!nvidia-smi
!python -c 'import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())'

In [ ]:
%%bash
pip install -q --upgrade pip
pip install -q "unsloth[colab-new]>=2025.12,<2026.06"
pip install -q "unsloth_zoo>=2025.12,<2026.06"
pip install -q "trl>=0.12,<0.16" "transformers>=4.48,<4.60" "peft>=0.14,<0.20" "accelerate>=1.2,<2.0"
pip install -q "datasets>=3.0,<4.0" "wandb>=0.18,<1.0" "bitsandbytes>=0.45" httpx fastapi uvicorn pydantic

## 1. Clone repo + boot the pool server

The pool server exposes our env as `/allocate /reset /exec_tool /evaluate /close`. We run it in the same Colab VM, in a background thread, so the GRPO rollout loop hits `http://127.0.0.1:8100`.

In [ ]:
import subprocess, os, time

if not os.path.exists('/content/sre-enginnerllm'):
    subprocess.check_call(['git', 'clone', 'https://github.com/dakshdoesdev/sre-enginnerllm.git', '/content/sre-enginnerllm'])
os.chdir('/content/sre-enginnerllm')
subprocess.check_call(['pip', 'install', '-q', '-e', '.'])

# Boot the pool server in the background
pool_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'openclaw_integration.pool_server:app',
     '--host', '127.0.0.1', '--port', '8100'],
    stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
)
time.sleep(5)
import httpx
print(httpx.get('http://127.0.0.1:8100/healthz', timeout=10).json())

## 2. Config

In [ ]:
import os

# --- Colab secrets bridge (HFTOKEN, WANDB) ---
try:
    from google.colab import userdata  # type: ignore
    for _src in ("HF_TOKEN", "HFTOKEN", "HUGGINGFACE_TOKEN"):
        try:
            v = userdata.get(_src)
            if v:
                os.environ["HF_TOKEN"] = v
                break
        except Exception:
            pass
    for _src in ("WANDB_API_KEY", "WANDB", "wandb"):
        try:
            v = userdata.get(_src)
            if v:
                os.environ["WANDB_API_KEY"] = v
                break
        except Exception:
            pass
except ImportError:
    pass

if "HF_TOKEN" not in os.environ:
    raise RuntimeError(
        "HF_TOKEN not set. Add `HFTOKEN` (or `HF_TOKEN`) in Colab Secrets "
        "(left sidebar 🔑 panel) and toggle 'Notebook access' on."
    )
# ---

SFT_ADAPTER_REPO = 'dakshdoesdev/sre-gym-qwen35-4b-sft'   # output of sanity_run.ipynb
GRPO_ADAPTER_REPO = 'dakshdoesdev/sre-gym-qwen35-4b-grpo' # destination

NUM_GRPO_STEPS = 300        # total policy updates
ROLLOUTS_PER_STEP = 4       # K (group size)
MAX_TICKS = 12              # per-rollout cap; matches env's scenario.max_ticks
LEARNING_RATE = 5e-6        # much lower than SFT — GRPO needs gentle updates
KL_COEFF = 0.05             # reserved for v2 (PPO-clip + KL)
PPO_CLIP = 0.2
SAVE_EVERY = 25             # push adapter to Hub every N steps
POOL_URL = 'http://127.0.0.1:8100'

os.environ.setdefault('WANDB_PROJECT', 'sre-gym-grpo')
os.environ.setdefault('WANDB_RUN_NAME', 'qwen35-4b-grpo-300')
os.environ.setdefault(
    'WANDB_MODE',
    'online' if os.environ.get('WANDB_API_KEY') else 'offline',
)

print(f"HF_TOKEN set:      {'yes' if os.environ.get('HF_TOKEN') else 'NO'}")
print(f"WANDB_API_KEY set: {'yes (online)' if os.environ.get('WANDB_API_KEY') else 'no (offline mode)'}")
print(f"SFT source:        {SFT_ADAPTER_REPO}")
print(f"GRPO destination:  {GRPO_ADAPTER_REPO}")

## 3. Load SFT checkpoint + LoRA for continued training

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_ADAPTER_REPO,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
# SFT adapter is already attached; just enable training mode.
model = FastLanguageModel.get_peft_model(
    model,
    r=32, lora_alpha=32, lora_dropout=0.0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42,
)

## 4. Rollout helper — play one episode with the current policy

In [ ]:
import json, httpx, uuid

SYSTEM_PROMPT = (
    'You are an SRE agent operating inside the sre-gym environment. '
    'On each turn, emit exactly one UnifiedIncidentAction JSON object — no surrounding prose, '
    'no code fences, no commentary. Valid action_types: query_logs, query_metrics, '
    'query_dependencies, query_deploys, rollback_deploy, restart_service, isolate_service, '
    'run_check, submit_hypothesis, escalate, declare_resolved.'
)

def generate_action(prompt_text, max_new_tokens=96):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to('cuda')
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    return text, inputs, out

def rollout_episode(scenario_id):
    """Play one full episode. Returns (trajectory, final_score)."""
    with httpx.Client(base_url=POOL_URL, timeout=60) as c:
        lease = c.post('/allocate', json={'task_key': scenario_id}).json()['lease_id']
        reset = c.post('/reset', json={'lease_id': lease, 'scenario_id': scenario_id}).json()
        obs = json.loads(reset['observation'])
        traj = []
        for tick in range(MAX_TICKS):
            if obs.get('done'):
                break
            prompt_text = obs['prompt_text']
            gen_text, inp_ids, out_ids = generate_action(prompt_text)
            # Parse action — on parse failure, let env return wrong_remediation_target
            try:
                action = json.loads(gen_text)
                tool_name = action.pop('action_type')
                args = action
            except Exception:
                tool_name, args = 'escalate', {}
            step_resp = c.post('/exec_tool', json={
                'lease_id': lease,
                'tool_call': {'name': tool_name, 'arguments': args},
            }).json()
            next_obs = json.loads(step_resp['observation'])
            traj.append({
                'prompt': prompt_text,
                'response': gen_text,
                'input_ids': inp_ids.cpu(),
                'output_ids': out_ids.cpu(),
                'reward': float(next_obs.get('reward') or 0.0),
            })
            obs = next_obs
        ev = c.post('/evaluate', json={'lease_id': lease}).json()
        c.post('/close', json={'lease_id': lease})
        return traj, float(ev['score'])

## 5. GRPO loss + update step

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE)
FastLanguageModel.for_training(model)

def per_token_logprobs(input_ids, output_ids):
    """Compute per-response-token logprobs for a (prompt, full_output) pair."""
    # full_seq = prompt tokens + response tokens
    full_seq = output_ids.to('cuda')
    prompt_len = input_ids.shape[-1]
    # Forward pass — we need logits for positions [prompt_len-1 .. end-1]
    logits = model(full_seq).logits[0]
    response_logits = logits[prompt_len - 1 : -1]  # [resp_len, vocab]
    response_tokens = full_seq[0, prompt_len:]      # [resp_len]
    logprobs = F.log_softmax(response_logits, dim=-1)
    token_logprobs = logprobs.gather(-1, response_tokens.unsqueeze(-1)).squeeze(-1)
    return token_logprobs

def grpo_step(trajectories, advantages):
    """Apply GRPO update to a group of trajectories.

    trajectories: list of list of turn dicts (one outer list per rollout)
    advantages: list[float] — one advantage per rollout, broadcast to all turns
    """
    optimizer.zero_grad()
    total_loss = 0.0
    n_turns = 0
    for traj, adv in zip(trajectories, advantages):
        for turn in traj:
            if turn['input_ids'] is None:
                continue
            token_lp = per_token_logprobs(turn['input_ids'], turn['output_ids'])
            # Simple policy gradient: -adv * mean_logprob (no PPO-clip baseline yet)
            turn_loss = -adv * token_lp.mean()
            total_loss = total_loss + turn_loss
            n_turns += 1
    if n_turns == 0:
        return 0.0
    loss = total_loss / n_turns
    loss.backward()
    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
    optimizer.step()
    return float(loss.detach().cpu())

## 6. Main GRPO loop

In [ ]:
import random, wandb

# Fetch scenarios from the pool server
scenarios = httpx.get(f'{POOL_URL}/healthz').json()['scenarios']
# Hold out 3 scenarios for eval (one per difficulty): drop procgen variants of these
HELD_OUT = {'worker_deploy_cascade__p04', 'db_config_rollout__p04', 'gateway_auth_rollout__p04'}
train_scenarios = [s for s in scenarios if s not in HELD_OUT]
print(f'training on {len(train_scenarios)} scenarios, holding out {len(HELD_OUT)}')

wandb.init(project=os.environ.get('WANDB_PROJECT', 'sre-gym-grpo'),
           name=os.environ.get('WANDB_RUN_NAME', 'qwen7b-grpo-300'))

running_reward = 0.0
for step in range(NUM_GRPO_STEPS):
    scenario = random.choice(train_scenarios)
    # Rollout K episodes in sequence (A100 40GB can't run K parallel 7B models)
    FastLanguageModel.for_inference(model)
    rollouts = []
    scores = []
    for k in range(ROLLOUTS_PER_STEP):
        traj, score = rollout_episode(scenario)
        rollouts.append(traj)
        scores.append(score)
    mean_score = sum(scores) / len(scores)
    advantages = [s - mean_score for s in scores]

    # Policy update
    FastLanguageModel.for_training(model)
    loss = grpo_step(rollouts, advantages)

    running_reward = 0.9 * running_reward + 0.1 * mean_score if step > 0 else mean_score
    wandb.log({
        'step': step, 'loss': loss, 'mean_score': mean_score,
        'running_reward': running_reward,
        'min_score': min(scores), 'max_score': max(scores),
        'adv_spread': max(advantages) - min(advantages),
        'scenario': scenario,
    })
    print(f'[{step:03d}] scenario={scenario:<35} scores={[f"{s:.2f}" for s in scores]} mean={mean_score:.3f} loss={loss:.4f}')

    if (step + 1) % SAVE_EVERY == 0:
        model.save_pretrained('/content/grpo_ckpt')
        tokenizer.save_pretrained('/content/grpo_ckpt')
        model.push_to_hub(GRPO_ADAPTER_REPO)
        print(f'[ckpt] pushed adapter at step {step+1} -> {GRPO_ADAPTER_REPO}')

## 7. Final push + held-out eval

In [ ]:
model.save_pretrained('/content/grpo_ckpt')
tokenizer.save_pretrained('/content/grpo_ckpt')
model.push_to_hub(GRPO_ADAPTER_REPO)

FastLanguageModel.for_inference(model)
eval_scores = {}
for scenario in HELD_OUT:
    rewards = []
    for _ in range(5):
        _, score = rollout_episode(scenario)
        rewards.append(score)
    eval_scores[scenario] = sum(rewards) / len(rewards)
print('held-out eval:')
for s, r in eval_scores.items():
    print(f'  {s}: {r:.3f}')
wandb.log({'eval_score_mean': sum(eval_scores.values()) / len(eval_scores)})
wandb.finish()

## Verification checklist

- [ ] Cell 3 (pool server) prints `{"ok": true, ...}` with scenarios listed
- [ ] Cell 5 (model load) succeeds without OOM
- [ ] Cell 8 (main loop) shows `running_reward` climbing from ~0.15 to > 0.55 over 300 steps
- [ ] `dakshdoesdev/sre-gym-qwen7b-grpo` has a new adapter pushed
- [ ] Final held-out eval mean > SFT baseline

## Caveats

- This is a **simple policy gradient** (no PPO-clip, no reference-model KL). Good enough for a first training curve. For the full GRPO-with-clip+KL, extend `grpo_step()` with a frozen reference copy of the SFT model and compute `min(ratio * adv, clip(ratio) * adv) - kl_coeff * kl`.
- **Sequential rollouts** are slow (K×episode_time). A more efficient implementation batches generation across K prompts at once.
- If `running_reward` stays flat after 100 steps, the policy isn't learning. Likely causes: (1) advantages are zero because all K rollouts tie at the max-tick-cap fallback, (2) LR too low, (3) SFT checkpoint is too overfit. Reduce ROLLOUTS_PER_STEP or increase exploration temperature first, then consider going back to SFT for more samples.